# Run all stages on a list of papers

End-to-end **gap-extraction funnel** — give it a folder of PDFs *or* a `texts.jsonl`
(`{id, text}` per line) and it runs:

1. **Stage A** — container-free `pdf_to_blocks` + `slice_section_aware` (keep Discussion/Conclusion-type sections)
2. **Stage B** — `cue_label` rules + a frozen **bge-small + logreg** head (cached `pipeline_head.joblib`)
3. **Stage C** — batched **`yandexgpt-5-lite`** LLM precision filter (optional; needs `.env` keys)

Output: a tidy `gaps` table (one row per extracted limitation/future-work sentence) + CSV/JSON.
Mirrors `scripts/run_extraction_pipeline.py` exactly.

In [ ]:
# ===== Setup + config =====
import os, sys, glob, time, json
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "gap2idea").exists())
sys.path.insert(0, str(ROOT / "src"))
try:
    from dotenv import load_dotenv; load_dotenv(ROOT / ".env")
except Exception:
    pass
import numpy as np, pandas as pd
from gap2idea.pipeline.gap_funnel import (pdf_to_blocks, slice_section_aware,
                                          cue_label, split_sentences, _context)

# ---- CONFIG (edit these) ----
INPUT_PDFS  = str(ROOT / "data" / "pdfs" / "*.pdf")   # a folder/glob of PDFs, or None
INPUT_TEXTS = None                                     # path to texts.jsonl {id,text}, or None
STAGE_C     = True            # run the Yandex LLM precision filter (needs YANDEX_* in .env)
DOMAIN      = "biomed"        # "biomed" -> validate_rct ; "general" -> validate (arXiv/CS)
LLM_MODEL   = "yandexgpt-5-lite"
THRESHOLD   = 0.5             # Stage B head threshold (lower = recall-first)
HEAD_CACHE  = ROOT / "data" / "pipeline_head.joblib"
TRAIN_JSONL = ROOT / "data" / "scibert_prep" / "train.jsonl"
OUT         = ROOT / "gaps.tsv"
ENCODER     = "BAAI/bge-small-en-v1.5"
TARGET      = {"limitation", "future_work"}
print("ROOT:", ROOT)

In [ ]:
# ===== Stage B head: load cached bge-small+logreg, or train once from train.jsonl =====
import joblib
from sentence_transformers import SentenceTransformer
enc = SentenceTransformer(ENCODER)
if HEAD_CACHE.exists():
    clf = joblib.load(HEAD_CACHE); print("loaded head:", HEAD_CACHE)
elif TRAIN_JSONL.exists():
    from sklearn.linear_model import LogisticRegression
    rows = [json.loads(l) for l in TRAIN_JSONL.read_text(encoding="utf-8").splitlines() if l.strip()]
    s = [r["sentence"] for r in rows]; y = [r["label"] for r in rows]
    X = enc.encode(s, normalize_embeddings=True, batch_size=256, show_progress_bar=False)
    clf = LogisticRegression(C=2.0, max_iter=2000, class_weight="balanced").fit(X, y)
    joblib.dump(clf, HEAD_CACHE); print("trained + cached head:", HEAD_CACHE)
else:
    clf = None; print("no head and no train set -> cue-rules only")

In [ ]:
# ===== Stage B: cue rules win on type; frozen head fills the rest =====
def stage_b(regions, threshold=THRESHOLD):
    flat = [(ri, si, s) for ri, r in enumerate(regions) for si, s in enumerate(r.sentences)]
    pred = {}
    if clf is not None and flat:
        X = enc.encode([s for _, _, s in flat], normalize_embeddings=True,
                       batch_size=256, show_progress_bar=False)
        proba = clf.predict_proba(X); classes = list(clf.classes_)
        for (ri, si, _), p in zip(flat, proba):
            j = int(np.argmax(p)); pred[(ri, si)] = (classes[j], float(p[j]))
    rows, seen = [], set()
    for ri, r in enumerate(regions):
        for si, s in enumerate(r.sentences):
            rule_t = cue_label(s); mt, mp = pred.get((ri, si), ("none", 0.0))
            if rule_t:                       gtype, source, conf = rule_t, "rule", 0.9
            elif mt in TARGET and mp >= threshold: gtype, source, conf = mt, "model", mp
            else:                            continue
            key = " ".join(s.lower().split())
            if key in seen: continue
            seen.add(key)
            rows.append(dict(gap_type=gtype, gap_sentence=s, source=source,
                             confidence=round(conf, 3), section_type=r.section_type,
                             paragraph_text=_context(r.sentences, si, w=1)))
    return rows

In [ ]:
# ===== Load papers (PDFs or texts.jsonl) =====
def load_papers():
    if INPUT_TEXTS:
        df = pd.read_json(INPUT_TEXTS, lines=True, dtype=False)
        for _, r in df.iterrows():
            blocks = r["blocks"] if isinstance(r.get("blocks"), list) else None
            yield str(r["id"]), str(r["text"]), blocks
        return
    paths = []
    for p in ([INPUT_PDFS] if isinstance(INPUT_PDFS, str) else (INPUT_PDFS or [])):
        paths += sorted(glob.glob(p)) if "*" in p or not os.path.isdir(p) else sorted(glob.glob(os.path.join(p, "*.pdf")))
    for path in paths:
        blocks = pdf_to_blocks(path)
        text = " ".join(b["text"] for b in blocks) if blocks else ""
        yield Path(path).stem, text, (blocks or None)

papers = list(load_papers())
print(f"{len(papers)} paper(s) to process")

In [ ]:
# ===== Stage A + B over the corpus =====
all_rows = []; t0 = time.time()
for pid, text, blocks in papers:
    if not text.strip():
        print(f"  {pid}: no text, skipped"); continue
    regions = slice_section_aware(text, blocks=blocks)        # Stage A
    rows = stage_b(regions)                                    # Stage B
    for r in rows: r["id"] = pid
    all_rows.extend(rows)
    print(f"  {pid}: {len(split_sentences(text))} sents -> "
          f"{sum(len(r.sentences) for r in regions)} kept -> {len(rows)} gaps (A+B)")
print(f"[A+B] {len(all_rows)} candidate gaps [{time.time()-t0:.0f}s]")

In [ ]:
# ===== Stage C: LLM precision filter (optional) =====
if STAGE_C and all_rows:
    os.environ.setdefault("LLM_PROVIDER", "yandex")
    from gap2idea.pipeline.gap_llm_filter import LLMGapFilter
    from gap2idea.pipeline.llm import get_llm_client
    folder = os.environ.get("YANDEX_FOLDER_ID", "").strip()
    model = f"gpt://{folder}/{LLM_MODEL}/latest" if folder else LLM_MODEL
    mode = "validate_rct" if DOMAIN == "biomed" else "validate"
    filt = LLMGapFilter(backend="api", model=model, mode=mode, client=get_llm_client("yandex"))
    n0 = len(all_rows); t1 = time.time()
    all_rows = filt.filter_gaps(all_rows, protect_rules=True, batch=True, use_context=True)
    print(f"[C] {mode}/{LLM_MODEL}: judged {filt.n_judged} in {filt.n_calls} calls, "
          f"kept {len(all_rows)}/{n0} [{time.time()-t1:.0f}s]")
else:
    print("Stage C skipped (STAGE_C=False or no candidates)")

In [ ]:
# ===== Results =====
cols = ["id", "gap_type", "gap_sentence", "section_type", "source", "confidence", "paragraph_text"]
df = pd.DataFrame(all_rows, columns=cols)
df.to_csv(OUT, sep="\t", index=False)
df.to_json(OUT.with_suffix(".json"), orient="records", force_ascii=False, indent=1)
print(f"wrote {OUT}  ({len(df)} gaps, {df['id'].nunique()} papers)")
print("by type:", df.groupby("gap_type").size().to_dict())
df.head(20)